# Uso de script de modularización

### Paso 1 — Carga del panel de ventanas electorales y Revisión de variables



In [1]:
import sys
import pandas as pd

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")  
from ml_models.cargar_panel import cargar_panel, columnas_candidatas 
from ml_models.lasso import *
NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel,f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 151 columnas
provincial: 12 filas x 151 columnas
nacional: 12 filas x 151 columnas


In [2]:
cols_vc_por_nivel = {}
corr_por_nivel = {}
for nivel in NIVELES: 
    df = paneles[nivel]
    cols_vc = columnas_candidatas(df, excluir_adicional=["delta_v"])

    corr = df[cols_vc].corr(method="pearson")  # pairwise, ignora NaN automáticamente
    cols_vc_por_nivel[nivel] = cols_vc
    corr_por_nivel[nivel] = corr
for nivel in NIVELES:
    print(f"{nivel}: {len(cols_vc)} variables candidatas, N={len(df)}")
corr_por_nivel["municipal"] 

municipal: 117 variables candidatas, N=12
provincial: 117 variables candidatas, N=12
nacional: 117 variables candidatas, N=12


,desocupacion_cobertura_parcial,desocupacion_delta_nivel,desocupacion_delta_pendiente,desocupacion_final_vc,desocupacion_nivel_vc,desocupacion_nivel_vl,desocupacion_pendiente_vc,desocupacion_pendiente_vl,desocupacion_volatilidad_vc,desocupacion_volatilidad_vl,...,tc_oficial_cobertura_parcial,tc_oficial_delta_nivel,tc_oficial_delta_pendiente,tc_oficial_final_vc,tc_oficial_nivel_vc,tc_oficial_nivel_vl,tc_oficial_pendiente_vc,tc_oficial_pendiente_vl,tc_oficial_volatilidad_vc,tc_oficial_volatilidad_vl
desocupacion_cobertura_parcial,1.000000,0.059879,0.365129,0.507663,0.490893,0.165688,-0.341839,0.047982,0.278257,-0.039538,...,NaN,-0.272743,-0.299844,-0.317495,-0.308755,-0.294031,-0.327083,-0.277194,-0.325106,-0.280969
desocupacion_delta_nivel,0.059879,1.000000,-0.704051,-0.287664,-0.477285,-0.777436,0.644708,0.995133,-0.407640,-0.847830,...,NaN,0.205776,0.110870,0.180213,0.197853,0.193566,0.144599,0.199828,0.147865,0.192945
desocupacion_delta_pendiente,0.365129,-0.704051,1.000000,0.678540,0.609044,0.667657,-0.278913,-0.721506,0.411142,0.517413,...,NaN,-0.013043,0.001606,-0.024382,-0.025642,-0.034030,-0.016657,-0.014155,-0.016543,-0.014519
desocupacion_final_vc,0.507663,-0.287664,0.678540,1.000000,0.967016,0.735748,-0.892786,-0.324685,0.485517,0.400098,...,NaN,-0.225549,-0.315856,-0.219954,-0.209513,-0.257800,-0.233651,-0.233141,-0.231804,-0.240848
desocupacion_nivel_vc,0.490893,-0.477285,0.609044,0.967016,1.000000,0.899991,-0.919701,-0.524422,0.663285,0.617906,...,NaN,-0.280850,-0.364126,-0.266020,-0.255274,-0.299475,-0.281650,-0.286523,-0.279971,-0.292257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tc_oficial_nivel_vl,-0.294031,0.193566,-0.034030,-0.257800,-0.299475,-0.333563,0.261991,0.193741,-0.102140,-0.193025,...,NaN,0.995892,0.978780,0.999631,0.999409,1.000000,0.993037,0.997464,0.993922,0.998647
tc_oficial_pendiente_vc,-0.327083,0.144599,-0.016657,-0.233651,-0.281650,-0.336658,0.213888,0.151610,-0.164768,-0.154158,...,NaN,0.980991,0.995946,0.995588,0.989649,0.993037,1.000000,0.984806,0.999966,0.988452
tc_oficial_pendiente_vl,-0.277194,0.199828,-0.014155,-0.233141,-0.286523,-0.329545,0.268812,0.197468,-0.102463,-0.213937,...,NaN,0.999782,0.967473,0.996561,0.999316,0.997464,0.984806,1.000000,0.986196,0.999747
tc_oficial_volatilidad_vc,-0.325106,0.147865,-0.016543,-0.231804,-0.279971,-0.336782,0.213485,0.154449,-0.163047,-0.157353,...,NaN,0.982549,0.995297,0.996304,0.990772,0.993922,0.999966,0.986196,1.000000,0.989660


In [3]:
chk = paneles["municipal"].set_index("id_transicion").loc["municipal_2015_2017", "ipc_nivel_vc"]
print("CHK ipc_nivel_vc municipal_2015_2017 =", chk)
assert abs(chk - 97.634) < 0.01, f"El notebook está leyendo el CSV viejo! valor={chk}"

CHK ipc_nivel_vc municipal_2015_2017 = 97.6340396754086


### Paso 2 — Sub-selección: colapsar clusters redundantes

La matriz de correlación mostró un cluster de colinealidad casi perfecta entre `ipc_*` y `tc_oficial_*` (r > 0.98 en todo el bloque), además de pares menores en `desocupacion`, `resultado_fiscal` y `salario_real`. Se busca simplificar reduciendo los datos que son redundantes para evitar que LASSO elija de forma aleatoria entre esas opciones.

**Parametrización fijada:**

| Decisión | Valor |
|---|---|
| Umbral de redundancia | `\|r\| ≥ 0.90` |
| Alcance | Transitivo (single-linkage): si A-B≥0.90 y B-C≥0.90, A/B/C van al mismo cluster aunque A-C no llegue al umbral |
| Desempate 1 | Sufijo `_nivel_vc` preferido sobre `_final`/`_pendiente`/`_volatilidad`/`_acum` |
| Desempate 2 | Prioridad teórica de la variable (`ipc` > `desocupacion` > `icg` > `icc` > `salario_real` > `tc_oficial` > `reservas` > `resultado_fiscal`) — orden de relevancia en la literatura de voto económico citada, no un criterio estadístico |

In [4]:
UMBRAL_REDUNDANCIA = 0.90
PRIORIDAD_TEORICA = ["ipc", "desocupacion", "icg", "icc", "salario_real", "tc_oficial", "reservas", "resultado_fiscal", "emae"]
ORDEN_SUFIJO = ["_nivel_vc", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_acum_vc"]

clusters_por_nivel = {}
columnas_finales_por_nivel = {}

for nivel in NIVELES:
    df = paneles[nivel]
    cols_vc = cols_vc_por_nivel[nivel]
    corr = corr_por_nivel[nivel]
    print(f"\n\nNivel: {nivel}")
    clusters_por_nivel[nivel] = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    columnas_finales_por_nivel[nivel] = [elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA) if len(cl) > 1 else next(iter(cl)) for cl in clusters_por_nivel[nivel]]

    print(f"De {len(cols_vc)} columnas candidatas, quedan {len(columnas_finales_por_nivel[nivel])} tras colapsar clusters (umbral={UMBRAL_REDUNDANCIA})\n")
    for cl in clusters_por_nivel[nivel]:
        if len(cl) > 1:
            elegido = elegir_representante(cl, df, ORDEN_SUFIJO, PRIORIDAD_TEORICA)
            print(f"cluster ({len(cl)}): {sorted(cl)} -> queda: {elegido}")



Nivel: municipal
De 117 columnas candidatas, quedan 73 tras colapsar clusters (umbral=0.9)

cluster (2): ['desocupacion_delta_nivel', 'desocupacion_pendiente_vl'] -> queda: desocupacion_delta_nivel
cluster (3): ['desocupacion_final_vc', 'desocupacion_nivel_vc', 'desocupacion_pendiente_vc'] -> queda: desocupacion_nivel_vc
cluster (10): ['desocupacion_nivel_vl', 'emae_final_vc', 'emae_nivel_vc', 'emae_nivel_vl', 'resultado_fiscal_nivel_vc', 'resultado_fiscal_nivel_vl', 'resultado_fiscal_volatilidad_vl', 'salario_real_final_vc', 'salario_real_nivel_vc', 'salario_real_nivel_vl'] -> queda: salario_real_nivel_vc
cluster (2): ['emae_delta_nivel', 'emae_pendiente_vl'] -> queda: emae_pendiente_vl
cluster (5): ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> queda: hacinamiento_medio_cobertura_parc

In [5]:
datos_final = {}
for nivel in NIVELES:
    X, y = construir_Xy_final(nivel, columnas_finales_por_nivel[nivel], paneles, target="delta_v")
    datos_final[nivel] = (X, y)
    print(f"{nivel}: N={len(y)}, P={X.shape[1]}")

[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']
municipal: N=10, P=67
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']
provincial: N=10, P=67
[nacional] excluye 2 fila(s) por NaN: ['nacional_2001_2003', 'nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']
nacional: N=10, P=66


In [6]:
faltantes = columnas_nan("nacional", "nacional_2013_2015", columnas_finales_por_nivel["nacional"], paneles)
print("Columna(s) que rompen nacional_2013_2015:", faltantes)

Columna(s) que rompen nacional_2013_2015: []


In [7]:
for nivel, id_t in [("municipal", "municipal_2001_2003"), ("provincial", "provincial_2001_2003")]:
    faltantes = columnas_nan(nivel, id_t, columnas_finales_por_nivel[nivel],paneles)
    print(f"{id_t}: NaN en -> {faltantes}")


municipal_2001_2003: NaN en -> ['desocupacion_delta_nivel', 'desocupacion_delta_pendiente', 'desocupacion_volatilidad_vl', 'emae_pendiente_vl', 'emae_delta_pendiente', 'emae_pendiente_vc', 'emae_volatilidad_vc', 'emae_volatilidad_vl', 'hacinamiento_medio_delta_nivel', 'hacinamiento_medio_nivel_vc', 'icc_delta_nivel', 'icc_delta_pendiente', 'icc_nivel_vl', 'icc_volatilidad_vl', 'icg_delta_nivel', 'icg_delta_pendiente', 'icg_nivel_vl', 'icg_volatilidad_vl', 'pct_hogares_ayuda_social_gobierno_delta_nivel', 'pct_hogares_ayuda_social_gobierno_nivel_vc', 'pct_hogares_prestamo_bancario_delta_nivel', 'pct_hogares_prestamo_bancario_nivel_vc', 'pct_hogares_vendio_pertenencias_delta_nivel', 'pct_hogares_vendio_pertenencias_nivel_vc', 'pct_sin_cobertura_salud_delta_nivel', 'pct_sin_cobertura_salud_nivel_vc', 'reservas_delta_nivel', 'reservas_delta_pendiente', 'reservas_nivel_vl', 'reservas_volatilidad_vl', 'resultado_fiscal_delta_nivel', 'resultado_fiscal_delta_pendiente', 'salario_real_delta_nive

In [8]:
X_df, y_ser = datos_final["municipal"]
print("CHK2 salario_real_nivel_vc en X_df:")
print(X_df["salario_real_nivel_vc"])

print("\nCHK2 salario_real_nivel_vc en paneles['municipal'] (directo):")
print(paneles["municipal"].set_index("id_transicion")["salario_real_nivel_vc"])

CHK2 salario_real_nivel_vc en X_df:
0     7562.884717
1     9284.498521
2    11249.949485
3    15899.633564
4    19023.335099
5    20581.492162
6    19654.302519
7    17876.372749
8    17371.718356
9    15350.778187
Name: salario_real_nivel_vc, dtype: float64

CHK2 salario_real_nivel_vc en paneles['municipal'] (directo):
id_transicion
municipal_2001_2003     7266.847626
municipal_2003_2005     6162.028586
municipal_2005_2007     7562.884717
municipal_2007_2009     9284.498521
municipal_2009_2011    11249.949485
municipal_2011_2013    15899.633564
municipal_2013_2015    19023.335099
municipal_2015_2017    20581.492162
municipal_2017_2019    19654.302519
municipal_2019_2021    17876.372749
municipal_2021_2023    17371.718356
municipal_2023_2025    15350.778187
Name: salario_real_nivel_vc, dtype: float64


### Paso 3 — LASSO por coordinate descent (implementación propia)

Formulación: `(1/2n)·‖y - Xβ‖² + α·‖β‖₁` (convención sklearn/glmnet). `X` estandarizada a mano (`ddof=0`), `y` centrada; intercepto = `media(y)`, no se penaliza.

`soft_threshold`: operador proximal de L1, da la selección de variables (coeficiente exactamente en cero si `|z| <= alpha`). `lasso_coordinate_descent`: actualiza una coordenada de `β` a la vez hasta que el cambio máximo entre iteraciones sea `< tol`.

In [9]:
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    X_std, medias, desvios = estandarizar(X_df)
    y_centrado = y_ser.values - y_ser.mean()
    n = len(y_centrado)

    alpha_prueba = 1.0
    beta_manual = lasso_coordinate_descent(X_std, y_centrado, alpha_prueba)

    kkt = verificar_kkt(X_std, y_centrado, beta_manual, alpha_prueba, n)
    print(f"{nivel} - KKT:", kkt)

    # Chequeo 2: alpha=0 debe coincidir con OLS
    beta_alpha_cero = lasso_coordinate_descent(X_std, y_centrado, alpha=0.0, max_iter=5000)
    beta_ols, *_ = np.linalg.lstsq(X_std, y_centrado, rcond=None)
    print(f"{nivel} - Máxima diferencia vs. OLS (alpha=0):", np.max(np.abs(beta_alpha_cero - beta_ols)))

    residuo_manual = y_centrado - X_std @ beta_alpha_cero
    residuo_ols = y_centrado - X_std @ beta_ols

    print(f"{nivel} - Residuo manual (debería ser ~0 si el sistema es subdeterminado):", np.max(np.abs(residuo_manual)))
    print(f"{nivel} - Residuo OLS (debería ser ~0 también):", np.max(np.abs(residuo_ols)))

municipal - KKT: {'error_max_en_activos': np.float64(8.968714316770487e-07), 'exceso_max_en_inactivos': np.float64(-0.1323353583734015), 'n_activos': np.int64(9)}
municipal - Máxima diferencia vs. OLS (alpha=0): 5.776169454217677
municipal - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 5.4400031146428773e-08
municipal - Residuo OLS (debería ser ~0 también): 2.842170943040401e-14
provincial - KKT: {'error_max_en_activos': np.float64(8.075512905048043e-07), 'exceso_max_en_inactivos': np.float64(-0.009507915327099381), 'n_activos': np.int64(9)}
provincial - Máxima diferencia vs. OLS (alpha=0): 4.982562264277203
provincial - Residuo manual (debería ser ~0 si el sistema es subdeterminado): 5.270944020141499e-08
provincial - Residuo OLS (debería ser ~0 también): 3.1807889655510735e-14
nacional - KKT: {'error_max_en_activos': np.float64(7.667250643272894e-07), 'exceso_max_en_inactivos': np.float64(-0.007691363089020742), 'n_activos': np.int64(9)}
nacional - Máxima diferenc

### Paso 4 — Grilla de alpha + LOO-CV manual

In [10]:
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: alpha_min=7.3325  alpha_1se=34.5719  (techo grilla=34.5719)
provincial: alpha_min=13.9468  alpha_1se=32.4954  (techo grilla=32.4954)
nacional: alpha_min=4.5198  alpha_1se=7.9437  (techo grilla=32.5287)


In [11]:

#validacion
resultados_cv = {}
for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    print(f"{nivel}: X_df.shape={X_df.shape}, len(y_ser)={len(y_ser)}, 'resultado_fiscal_final_vc' in cols: {'resultado_fiscal_final_vc' in X_df.columns}")
    resultados_cv[nivel] = lasso_loocv_manual(X_df, y_ser, factor_extension=3.0)
    r = resultados_cv[nivel]
    print(f"{nivel}: alpha_min={r['alpha_min']:.4f}  alpha_1se={r['alpha_1se']:.4f}  (techo grilla={r['alphas'][-1]:.4f})")

municipal: X_df.shape=(10, 67), len(y_ser)=10, 'resultado_fiscal_final_vc' in cols: True
municipal: alpha_min=7.3325  alpha_1se=34.5719  (techo grilla=34.5719)
provincial: X_df.shape=(10, 67), len(y_ser)=10, 'resultado_fiscal_final_vc' in cols: True
provincial: alpha_min=13.9468  alpha_1se=32.4954  (techo grilla=32.4954)
nacional: X_df.shape=(10, 66), len(y_ser)=10, 'resultado_fiscal_final_vc' in cols: True
nacional: alpha_min=4.5198  alpha_1se=7.9437  (techo grilla=32.5287)


In [12]:
for nivel in ["provincial", "nacional"]:
    X_df, y_ser = datos_final[nivel]
    print(f"--- {nivel} ---")
    print(verificar_saturacion(X_df, y_ser, factores=[1, 3, 10]))
    print()

--- provincial ---
   factor_extension       techo  alpha_min   alpha_1se     mse_min  \
0                 1   10.831814  10.831814   10.831814  258.824892   
1                 3   32.495443  13.946831   32.495443  245.536501   
2                10  108.318144  15.050757  108.318144  245.536501   

   mse_en_techo  
0    258.824892  
1    245.536501  
2    245.536501  

--- nacional ---
   factor_extension       techo  alpha_min  alpha_1se     mse_min  \
0                 1   10.842893   4.653698   8.178924  123.591595   
1                 3   32.528678   4.519845   7.943676  123.893589   
2                10  108.428926   4.877602   7.445260  123.890854   

   mse_en_techo  
0    193.234874  
1    190.419842  
2    190.419842  



### Paso 6 — Ajuste final: coeficientes y mejora sobre baseline trivial

`baseline_trivial_loocv`: MSE en LOO de predecir el promedio de los demás puntos -- piso de comparación. `mse_en_alpha`: mismo esquema de LOO que `lasso_loocv_manual`, pero para un alpha puntual (se recalcula, no se guarda en la función de CV).

**Resultado:**

| Nivel | Mejora de `alpha_min` sobre baseline | Mejora de `alpha_1se` |
|---|---|---|
| Municipal | +15.01% | +0.0% |
| Provincial | +0.0% | +0.0% |
| Nacional | +34.94% | +19.38% |

Coeficientes en `alpha_min`: `icg_delta_pendiente` (1.57) e `icg_pendiente_vc` (2.88) en municipal; `icg_pendiente_vc` (6.32) en nacional; provincial no selecciona ninguna variable. En `alpha_1se`: solo `icg_pendiente_vc` (2.90) en nacional, nada en municipal ni provincial.

In [13]:
resumen = []
coeficientes_min, coeficientes_1se = {}, {}

for nivel in NIVELES:
    X_df, y_ser = datos_final[nivel]
    r = resultados_cv[nivel]

    base = baseline_trivial_loocv(y_ser)
    mse_min = mse_en_alpha(X_df, y_ser, r["alpha_min"])
    mse_1se = mse_en_alpha(X_df, y_ser, r["alpha_1se"])

    resumen.append({
        "nivel": nivel,
        "baseline_mse": base,
        "mejora_alpha_min_%": 100 * (1 - mse_min / base),
        "mejora_alpha_1se_%": 100 * (1 - mse_1se / base),
    })

    coeficientes_min[nivel] = ajustar_final(X_df, y_ser, r["alpha_min"])
    coeficientes_1se[nivel] = ajustar_final(X_df, y_ser, r["alpha_1se"])

tabla_resumen = pd.DataFrame(resumen).set_index("nivel")
print(tabla_resumen)

            baseline_mse  mejora_alpha_min_%  mejora_alpha_1se_%
nivel                                                           
municipal     251.482226           15.008350            0.000000
provincial    245.536501            0.000000            0.000000
nacional      190.419842           34.936618           19.381467


In [14]:
tabla_coef_min = pd.DataFrame(coeficientes_min)
tabla_coef_min = tabla_coef_min[(tabla_coef_min != 0).any(axis=1)]
print("Coeficientes distintos de cero (alpha_min):")
tabla_coef_min

Coeficientes distintos de cero (alpha_min):


,municipal,provincial,nacional
icg_delta_pendiente,1.567111,0.0,0.000000
icg_pendiente_vc,2.880080,0.0,6.323047
icg_volatilidad_vl,0.000000,0.0,NaN


### Chequeo de estabilidad (leave-one-transition-out) — los tres niveles

**Municipal (alpha_1se=34.57):** ninguna variable sobrevive en ninguna de las 10 corridas.

**Provincial (alpha=13.947, criterio alpha_min):** ninguna variable sobrevive en ninguna de las 10 corridas.

**Nacional (alpha=4.520, criterio alpha_min):** `icg_pendiente_vc` sobrevive en las 10/10 corridas (el único predictor estable de todo `01_1`); el resto aparece de forma intermitente: `icg_delta_pendiente` (1/10), `icg_final_vc` (1/10), `desocupacion_delta_pendiente` (1/10), `reservas_delta_pendiente` (1/10), `reservas_volatilidad_vc` (3/10), `salario_real_delta_pendiente` (1/10), `tasa_informalidad_pendiente_vl` (2/10), `icc_delta_pendiente` (1/10), `tasa_informalidad_volatilidad_vc` (1/10) -- ninguna de estas últimas sobrevive en más de 3 de las 10 corridas.

In [15]:
ALPHA_PARA_ESTABILIDAD = {
    "municipal": ("alpha_1se", resultados_cv["municipal"]["alpha_1se"]),
    "provincial": ("alpha_min", resultados_cv["provincial"]["alpha_min"]),
    "nacional": ("alpha_min", resultados_cv["nacional"]["alpha_min"]),
}

for nivel in NIVELES:
    df = paneles[nivel]
    criterio, alpha = ALPHA_PARA_ESTABILIDAD[nivel]
    X_df, y_ser = datos_final[nivel] 
    resultado = estabilidad_seleccion(nivel, alpha, df, columnas_finales_por_nivel[nivel], "delta_v", X_df, y_ser)
    sobrevivientes = resultado.loc[:, (resultado != 0).any(axis=0)]

    print(f"--- {nivel} (alpha={alpha:.3f}, criterio={criterio}) ---")
    if sobrevivientes.empty:
        print("Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.\n")
    else:
        print(sobrevivientes)
        print()

--- municipal (alpha=34.572, criterio=alpha_1se) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- provincial (alpha=13.947, criterio=alpha_min) ---
Ninguna variable sobrevive en ninguna de las corridas leave-one-transition-out.

--- nacional (alpha=4.520, criterio=alpha_min) ---
                        desocupacion_delta_pendiente  icc_delta_pendiente  \
sin_nacional_2005_2007                      0.000000             0.000000   
sin_nacional_2007_2009                      0.000000             0.000000   
sin_nacional_2009_2011                      0.000000             0.000000   
sin_nacional_2011_2013                      0.000000             0.000000   
sin_nacional_2013_2015                      0.000000             0.000000   
sin_nacional_2015_2017                      0.000000             0.132634   
sin_nacional_2017_2019                      0.000000             0.000000   
sin_nacional_2019_2021                     -0.049611            

In [16]:
import pandas as pd
df = pd.read_csv("/workspaces/analisis-politica-economia/data/tfi_data/panel_ventanas.csv")
df[df.id_transicion=="municipal_2015_2017"]["ipc_nivel_vc"].values

array([97.63403968])